# 03 - Train the ReID encoder

| Setting | Value |
| --- | --- |
| Internet | **Off** |
| Accelerator | **GPU** |

Identity persistence, not activity recognition, is the hard problem in this system. Every
behavioural claim downstream is attributed to a person, so a single false merge corrupts two
residents' baselines at once and does so silently.

This notebook therefore reports three numbers that standard ReID papers usually omit:

| Metric | Why it is the one that matters here |
| --- | --- |
| `rank1_at_short_side_96` | Home CCTV crops are far smaller than Market-1501 crops. |
| `false_merge_rate` | Confusing two residents is the failure that poisons baselines. |
| `clothing_change_rank1` | Residents wear the same clothes for days, then change. |

Standard mAP is logged too, but it is not the acceptance gate.


In [ ]:
import json, socket, sys
from pathlib import Path

import numpy as np
import torch

assert torch.cuda.is_available(), 'Enable the GPU accelerator in the notebook settings.'
print('gpu:', torch.cuda.get_device_name(0))

def has_internet(host='raw.githubusercontent.com', port=443, timeout=3):
    try:
        socket.create_connection((host, port), timeout=timeout)
        return True
    except OSError:
        return False

print('internet:', 'ENABLED - turn it off' if has_internet() else 'disabled (correct)')

DATASET_DIR = Path('/kaggle/input/reid-lowres-v1')
RUN_DIR = Path('/kaggle/working/runs/reid-001')
assert DATASET_DIR.exists(), f'attach the prepared ReID dataset at {DATASET_DIR}'

sys.path.insert(0, str(DATASET_DIR / 'code' / 'src'))
sys.path.insert(0, str(DATASET_DIR / 'code' / 'train'))

manifest = json.loads((DATASET_DIR / 'manifest.json').read_text())
print(f"dataset: {manifest['name']} v{manifest['version']}, {manifest['n_records']} crops")


In [ ]:
from _offline_tracker import OfflineTracker
from train_reid import seed_everything

SEED = 42
EPOCHS = 40
TRAIN_SHORT_SIDE = 128  # matches the --train-short-side default in train_reid.py
EVAL_SHORT_SIDES = [128, 96, 64]
IDS_PER_BATCH = 8
CROPS_PER_ID = 4  # PK sampling: triplet loss needs positives inside every batch
EMBED_DIM = 256
LR = 3e-4

seed_everything(SEED)
records = manifest['records']
identities = sorted({r['subject_id'] for r in records if r.get('split') == 'train'})
print(f'{len(identities)} training identities, batch = {IDS_PER_BATCH * CROPS_PER_ID} crops')
print('NOTE: identities must not appear in both train and gallery splits.')


## PK sampling and resolution-aware augmentation

Two choices here are specific to elderly home monitoring:

1. **Random downscale then upscale back** during training. The encoder must produce stable
   embeddings for a 64px crop and a 128px crop of the same person, because distance to the
   camera varies constantly in a home.
2. **Colour jitter is kept mild.** Aggressive jitter is standard in ReID benchmarks, but here
   clothing colour is one of the few reliable cues within a single day, so destroying it
   removes signal the deployed gallery depends on.


In [ ]:
from collections import defaultdict

from torch.utils.data import DataLoader, Dataset, Sampler

ASPECT = 0.5  # person crops are roughly 2:1 tall

def load_crop(path, short_side, train):
    height = int(short_side / ASPECT)
    full = Path(path)
    if full.exists():
        import cv2
        image = cv2.cvtColor(cv2.imread(str(full)), cv2.COLOR_BGR2RGB)
        if train and np.random.rand() < 0.5:
            # Degrade then restore size: teaches resolution invariance, not detail invention.
            factor = np.random.choice([2, 3, 4])
            small = cv2.resize(image, (max(8, image.shape[1] // factor), max(8, image.shape[0] // factor)))
            image = cv2.resize(small, (image.shape[1], image.shape[0]), interpolation=cv2.INTER_LINEAR)
        image = cv2.resize(image, (short_side, height))
    else:
        image = np.random.randint(0, 255, (height, short_side, 3), dtype=np.uint8)
    array = image.astype(np.float32) / 255.0
    if train:
        if np.random.rand() < 0.5:
            array = array[:, ::-1]
        array = np.clip(array * np.random.uniform(0.85, 1.15), 0.0, 1.0)
        if np.random.rand() < 0.3:
            # Random erasing stands in for the occlusion that furniture causes constantly.
            h, w = array.shape[:2]
            eh, ew = np.random.randint(h // 8, h // 3), np.random.randint(w // 8, w // 3)
            top, left = np.random.randint(0, h - eh), np.random.randint(0, w - ew)
            array[top:top + eh, left:left + ew] = np.random.rand()
    return torch.from_numpy(np.ascontiguousarray(array)).permute(2, 0, 1)

class CropDataset(Dataset):
    def __init__(self, records, root, split, short_side, train):
        self.records = [r for r in records if r.get('split') == split]
        self.root = Path(root)
        self.short_side = short_side
        self.train = train
        self.ids = sorted({r['subject_id'] for r in self.records})
        self.id_to_index = {name: i for i, name in enumerate(self.ids)}
        self.by_id = defaultdict(list)
        for index, record in enumerate(self.records):
            self.by_id[record['subject_id']].append(index)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        tensor = load_crop(self.root / record['path'], self.short_side, self.train)
        return tensor, self.id_to_index[record['subject_id']]

class PKSampler(Sampler):
    """Yield P identities x K crops per batch so triplet loss always has valid positives."""

    def __init__(self, dataset, ids_per_batch, crops_per_id, batches):
        self.dataset, self.p, self.k, self.batches = dataset, ids_per_batch, crops_per_id, batches

    def __iter__(self):
        eligible = [i for i in self.dataset.ids if len(self.dataset.by_id[i]) >= 2]
        for _ in range(self.batches):
            for identity in np.random.choice(eligible, size=min(self.p, len(eligible)), replace=False):
                pool = self.dataset.by_id[identity]
                chosen = np.random.choice(pool, size=self.k, replace=len(pool) < self.k)
                yield from (int(c) for c in chosen)

    def __len__(self):
        return self.batches * self.p * self.k

train_set = CropDataset(records, DATASET_DIR, 'train', TRAIN_SHORT_SIDE, train=True)
batches_per_epoch = max(1, len(train_set) // (IDS_PER_BATCH * CROPS_PER_ID))
train_loader = DataLoader(
    train_set,
    batch_size=IDS_PER_BATCH * CROPS_PER_ID,
    sampler=PKSampler(train_set, IDS_PER_BATCH, CROPS_PER_ID, batches_per_epoch),
    num_workers=2,
    drop_last=True,
)
print(f'{len(train_set)} train crops, {batches_per_epoch} batches/epoch')


In [ ]:
# OSNet-style encoder if torchreid is mounted, otherwise a compact residual CNN. Either way the
# output is an L2-normalised embedding, which is what wellbeing.perception.identity expects.
import torch.nn as nn
import torch.nn.functional as F

class Encoder(nn.Module):
    def __init__(self, embed_dim, num_classes):
        super().__init__()
        def block(cin, cout, stride):
            return nn.Sequential(
                nn.Conv2d(cin, cout, 3, stride=stride, padding=1, bias=False),
                nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                nn.Conv2d(cout, cout, 3, padding=1, bias=False),
                nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            )
        self.trunk = nn.Sequential(block(3, 64, 2), block(64, 128, 2), block(128, 256, 2), block(256, 384, 2))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.neck = nn.Sequential(nn.Flatten(), nn.Linear(384, embed_dim), nn.BatchNorm1d(embed_dim))
        self.classifier = nn.Linear(embed_dim, num_classes, bias=False)

    def forward(self, x):
        embedding = self.neck(self.pool(self.trunk(x)))
        # Classifier logits train on the unnormalised embedding; retrieval uses the normalised one.
        return F.normalize(embedding, dim=1), self.classifier(embedding)

model = Encoder(EMBED_DIM, len(train_set.ids)).cuda()
print('parameters:', sum(p.numel() for p in model.parameters()) / 1e6, 'M')


In [ ]:
# Train: batch-hard triplet + ID cross-entropy, the combination that holds up under occlusion.
tracker = OfflineTracker(RUN_DIR, {
    'task': 'person_reid',
    'dataset': manifest['name'],
    'dataset_version': manifest['version'],
    'train_short_side': TRAIN_SHORT_SIDE,
    'embed_dim': EMBED_DIM,
    'hyperparameters': {'seed': SEED, 'epochs': EPOCHS, 'p': IDS_PER_BATCH, 'k': CROPS_PER_ID, 'lr': LR},
})

MARGIN = 0.3

def batch_hard_triplet(embeddings, targets, margin=MARGIN):
    distances = torch.cdist(embeddings, embeddings)
    same = targets.unsqueeze(0) == targets.unsqueeze(1)
    eye = torch.eye(len(targets), dtype=torch.bool, device=targets.device)
    hardest_positive = (distances * (same & ~eye)).max(1).values
    hardest_negative = distances.masked_fill(same, float('inf')).min(1).values
    return F.relu(hardest_positive - hardest_negative + margin).mean()

identity_loss = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler()

for epoch in range(EPOCHS):
    model.train()
    totals = {'triplet': 0.0, 'identity': 0.0}
    for crops, targets in train_loader:
        crops, targets = crops.cuda(non_blocking=True), targets.cuda(non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast():
            embeddings, logits = model(crops)
            triplet = batch_hard_triplet(embeddings.float(), targets)
            identity = identity_loss(logits, targets)
            loss = triplet + identity
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        totals['triplet'] += float(triplet)
        totals['identity'] += float(identity)
    scheduler.step()
    steps = max(1, len(train_loader))
    tracker.log(epoch, triplet=totals['triplet'] / steps, identity=totals['identity'] / steps)
    if epoch % 5 == 0 or epoch == EPOCHS - 1:
        print(f"epoch {epoch:02d}  triplet {totals['triplet'] / steps:.4f}  id {totals['identity'] / steps:.4f}")

torch.save({'model': model.state_dict(), 'embed_dim': EMBED_DIM}, RUN_DIR / 'encoder.pt')
print('saved', RUN_DIR / 'encoder.pt')


## Evaluation across resolutions, and the false-merge audit

The false-merge rate is measured at the **deployed acceptance threshold** (0.72 in
`configs/default.yaml`), not at rank-1. Rank-1 always returns a nearest neighbour; the deployed
resolver refuses to answer below threshold, and that refusal is the safety property being
tested here.


In [ ]:
ACCEPT_THRESHOLD = 0.72  # keep in sync with identity.gallery accept threshold in configs/default.yaml

@torch.no_grad()
def embed_split(split, short_side):
    dataset = CropDataset(records, DATASET_DIR, split, short_side, train=False)
    if len(dataset) == 0:
        return None, None, dataset
    loader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=2)
    model.eval()
    vectors, ids = [], []
    with torch.cuda.amp.autocast():
        for crops, targets in loader:
            embeddings, _ = model(crops.cuda())
            vectors.append(embeddings.float().cpu())
            ids.append(targets)
    return torch.cat(vectors), torch.cat(ids), dataset

results = {}
for short_side in EVAL_SHORT_SIDES:
    query, query_ids, query_set = embed_split('test', short_side)
    if query is None:
        print(f'no test split; skipping {short_side}px')
        continue
    similarity = query @ query.T
    similarity.fill_diagonal_(-2.0)  # never retrieve the query itself
    best = similarity.argmax(1)
    best_similarity = similarity.max(1).values

    correct = (query_ids[best] == query_ids)
    rank1 = float(correct.float().mean())

    # A false merge is an ACCEPTED match to the wrong identity. Rejections are safe, not errors.
    accepted = best_similarity >= ACCEPT_THRESHOLD
    false_merges = int((accepted & ~correct).sum())
    false_merge_rate = false_merges / max(1, int(accepted.sum()))
    abstain_rate = float((~accepted).float().mean())

    results[short_side] = {
        'rank1': rank1,
        'false_merge_rate': false_merge_rate,
        'false_merges': false_merges,
        'accepted': int(accepted.sum()),
        'abstain_rate': abstain_rate,
    }
    print(f'{short_side:>4}px  rank1 {rank1:.3f}  false-merge {false_merge_rate:.4f} '
          f'({false_merges}/{int(accepted.sum())} accepted)  abstain {abstain_rate:.3f}')


In [ ]:
# Clothing-change check. Requires an `outfit_id` field in the manifest; if your source dataset
# lacks it, this is the single most valuable annotation to add, because a resident changing
# clothes is a daily event and appearance-only ReID fails on it every time.
has_outfit = any('outfit_id' in r for r in records)
clothing_rank1 = None

if has_outfit and 128 in results:
    query, query_ids, query_set = embed_split('test', 128)
    outfits = [query_set.records[i].get('outfit_id') for i in range(len(query_set))]
    similarity = query @ query.T
    similarity.fill_diagonal_(-2.0)
    # Mask out same-outfit gallery entries: force cross-outfit retrieval only.
    for i, outfit in enumerate(outfits):
        for j, other in enumerate(outfits):
            if outfit == other:
                similarity[i, j] = -2.0
    valid = similarity.max(1).values > -2.0
    if int(valid.sum()) > 0:
        best = similarity.argmax(1)
        clothing_rank1 = float((query_ids[best] == query_ids)[valid].float().mean())
        print(f'clothing-change rank1: {clothing_rank1:.3f} over {int(valid.sum())} queries')
else:
    print('no outfit_id annotations: clothing-change robustness is UNMEASURED.')
    print('Until it is measured, keep body-appearance prototypes on a short TTL (36h in config)')
    print('and do not let appearance alone confirm an identity.')

gate_rank1 = results.get(96, {}).get('rank1')
gate_merge = results.get(96, {}).get('false_merge_rate')
tracker.summarise(
    by_short_side={str(k): v for k, v in results.items()},
    rank1_at_short_side_96=gate_rank1,
    false_merge_rate=gate_merge,
    clothing_change_rank1=clothing_rank1,
    accept_threshold=ACCEPT_THRESHOLD,
    exit_criteria={
        'false_merge_rate_max': 0.001,
        'met': bool(gate_merge is not None and gate_merge <= 0.001),
    },
)
if gate_merge is not None:
    print()
    print(f'exit criterion false-merge <= 0.1%: {"MET" if gate_merge <= 0.001 else "NOT MET"}')


## Reading these numbers

- **High rank-1, high false-merge.** The embedding ranks well but its similarity scale does not
  match the deployed threshold. Raise `identity.gallery.accept`; a higher abstain rate is the
  correct trade, because `IdentityAssignment` already degrades gracefully when it abstains.
- **Rank-1 collapses from 128px to 64px.** Add more downscale augmentation, or gate ReID by
  crop size using `CropQuality.is_gallery_eligible` rather than trusting small crops.
- **Clothing-change rank-1 far below same-outfit rank-1.** Expected, and the reason the design
  fuses gait and spatiotemporal priors instead of relying on appearance.

Export the checkpoint as a Kaggle dataset and point `perception.reid.weights` at it. Do not
enable it in production until the false-merge gate passes: an unidentified resident produces a
missing data point, while a merged resident produces confident nonsense in two people's records.
